# SmolEdgeVLM retrained on broader data (COCO + TextCaps + VizWiz)

Same model, same size, same board code: only the numbers inside R, the connector and the LoRA change. SigLIP-B/16 and
SmolLM2-135M stay frozen (the LoRA is merged into SmolLM2 at the end, as before).

| Stage | What is trained | Data | Recipe |
|---|---|---|---|
| 1 | R (0.30 M, label-free) | 3,000 images, 1,000 from each source, with image changes | unchanged: one pass, 375 steps, batch 8, lr 1e-3 |
| 2 | connector (0.78 M), new | all training images, one random caption each time, with image changes | 1 epoch, batch 16, lr 1e-3, cosine |
| 3 | LoRA on SmolLM2 (4.88 M), new | the same images | 1 epoch, batch 16, lr 2e-4, rank 16 |

Training data (no image from any test set):
- **COCO train2014**: about 82,783 images, 5 captions each (val2014 is not used at all, so the Karpathy test split,
  the 500 held-out images and the 100 board images stay unseen)
- **TextCaps train**: 21,953 images, 5 captions each, captions that read text in the image
- **VizWiz-Captions train**: about 23,000 photos taken by blind users; pre-written "quality issues" captions and
  rejected captions are removed. Its validation split is not used (kept for the phase 2 test).

Captions: first sentence only, and only captions of at most 20 tokens are used (if an image has none, its shortest
caption is cut to 20 tokens). This keeps the learned captions short, so the caption time on the board stays the same.

Image changes during training (30% of images, one change each): darker, blur, noise, JPEG compression, lower resolution.

At the end, both new models (with and without LoRA) are compared with the current ones on 500 images from each of:
the COCO held-out set (first 100 = the board images), nocaps validation and TextCaps validation (unseen by every model,
TextCaps validation is in-domain for the new ones only). Full benchmarks: separate notebook.

**Inputs (+ Add Input):**
1. COCO 2014 (`jeffaudi/coco-2014-dataset-for-yolov3`)
2. The dataset made from `Code/Baselines-Kaggle/smoledgevlm_seed0_inputs.zip` (`long_M256.pt` is reused for the
   full-encoder mode; the other two files rebuild the current model for the comparison)
3. Optional: the current LoRA board package (dataset from `Jetson-Runs/ours_lora_jetson.zip`), to compare against
4. Only when resuming: this notebook's previous output

**Settings:** GPU T4, Internet ON, Persistence off. **Time:** about 6 to 9 hours (data preparation about 1 hour).
If a session ends early, add this notebook's output as an input and run all again: finished stages are loaded,
and an unfinished stage continues from its last checkpoint (every 1,000 steps).

**Output:** `retrain_check.json` (small, download first), `smoledgevlm_v2_jetson.zip` (board package, both models),
`ckpt/` (trained parts).

In [ ]:
!pip install -q "transformers==4.57.0" peft pycocoevalcap huggingface_hub pyarrow
!pip uninstall -y -q torchao   # Kaggle's torchao is older than peft accepts; nothing here needs it
import os, shutil, subprocess
J8 = '/usr/lib/jvm/java-8-openjdk-amd64/bin'          # the PTB tokenizer used for CIDEr needs Java
if not os.path.isdir(J8) and shutil.which('java') is None:
    subprocess.run('apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-8-jre-headless > /dev/null', shell=True)
if os.path.isdir(J8):
    os.environ['PATH'] = J8 + ':' + os.environ['PATH']
print('java:', shutil.which('java'))

In [ ]:
%%writefile /kaggle/working/jetson_foremerge.py
#!/usr/bin/env python3
"""ForeMerge on the Jetson Nano: full encoder vs ForeMerge, captioned and timed.

Needs only Python 3.6+, PyTorch 1.10+, NumPy and Pillow (no transformers). Same maths as foremerge.py:
frozen SigLIP-B/16@512, merge 1024 -> 256 tokens after layer 3 (fixed lattice, 7 x 7 window), restoration
module R, layers 4-12 with a log(size) attention bias, unmerge, pooling connector, frozen SmolLM2-135M with a
KV cache and greedy decoding.

    python3 jetson_foremerge.py --images images
    python3 jetson_foremerge.py --images images --runs 3 --max-new 24 --dtype float16 --out jetson_results

Inputs (same folder by default): foremerge_jetson_weights.npz and foremerge_jetson_meta.json, made on a PC
with export_for_jetson.py. Outputs: <out>.csv (one row per image and pipeline) and <out>.json (summary).
"""
import argparse, glob, json, math, os, resource, time
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image


# ------------------------------------------------------------------ loading
torch.backends.cudnn.enabled = False                # nothing here needs cuDNN; not loading it saves memory


def load_model(weights, meta_path, device, dtype):
    meta = json.load(open(meta_path))
    W = {}
    z = np.load(weights)
    for k in z.files:                              # one tensor at a time: keeps peak memory low on 2 GB
        W[k] = torch.from_numpy(z[k]).to(device=device, dtype=dtype)
    z.close()
    if device == 'cuda':
        torch.cuda.empty_cache()
    return meta, W


def mem_mb():
    """Peak process memory (MB) and currently available system memory (MB)."""
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
    avail = float('nan')
    try:
        for line in open('/proc/meminfo'):
            if line.startswith('MemAvailable:'):
                avail = int(line.split()[1]) / 1024.0
    except IOError:
        pass
    return peak, avail


# ------------------------------------------------------------------ building blocks
def gelu_tanh(x):                                  # SigLIP's gelu_pytorch_tanh (F.gelu has no "approximate" in torch 1.10)
    return 0.5 * x * (1.0 + torch.tanh(0.7978845608028654 * (x + 0.044715 * x * x * x)))


def rms_norm(x, w, eps):                           # LlamaRMSNorm: normalise in float32, scale in the model dtype
    xf = x.float()
    xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)
    return w * xf.to(x.dtype)


class ForeMergeJetson(object):
    def __init__(self, weights, meta_path, device='cuda', dtype=torch.float16):
        self.device, self.dtype = device, dtype
        self.meta, self.W = load_model(weights, meta_path, device, dtype)
        v, l, fm = self.meta['vision'], self.meta['lm'], self.meta['foremerge']
        self.v, self.l = v, l
        self.k, self.M, self.radius = fm['k'], fm['M'], fm['radius']
        self.grid = v['image_size'] // v['patch']
        self.N = self.grid * self.grid
        idx = np.arange(self.N)
        rows, cols = idx // self.grid, idx % self.grid
        self.coords = torch.from_numpy(np.stack([rows, cols], 1)).to(device)
        lattice = np.where((rows % 2 == 0) & (cols % 2 == 0), 2.0, np.where((rows % 2 == 1) & (cols % 2 == 1), 1.0, 0.0))
        score = torch.from_numpy(lattice - 1e-6 * idx)
        self.anchors = torch.topk(score, self.M).indices.sort().values.to(device)
        is_anc = torch.zeros(self.N, dtype=torch.bool, device=device)
        is_anc[self.anchors] = True
        self.others = (~is_anc).nonzero().squeeze(1)
        dist = (self.coords[self.others][:, None, :] - self.coords[self.anchors][None, :, :]).abs().max(-1)[0]
        self.outside = dist > self.radius                       # [768, 256]: anchors a patch may not join
        self.no_local = self.outside.all(1)                     # always False with the lattice; kept as in foremerge.py
        hd = l['hidden'] // l['heads']
        self.hd = hd
        self.inv_freq = 1.0 / (l['rope_theta'] ** (torch.arange(0, hd, 2, device=device).float() / hd))
        self.prompt = self.W['lm.embed'][torch.tensor(self.meta['prompt_ids'], device=device)][None]
        self.eos = self.meta['eos_id']
        self.special = set(self.meta['special_ids'])
        self.id_bytes = [bytes.fromhex(h) for h in self.meta['id_bytes_hex']]
        mean = torch.tensor(v['mean']).view(1, 3, 1, 1)
        std = torch.tensor(v['std']).view(1, 3, 1, 1)
        self.mean, self.std = mean.to(device), std.to(device)

    # ---------------------------------------------------------- image
    def pixels(self, img):
        """SigLIP preprocessing: bicubic resize to 512 x 512, scale to [0, 1], normalise with mean/std 0.5."""
        h, w = self.v['resize']
        x = np.asarray(img.convert('RGB').resize((w, h), Image.BICUBIC), dtype=np.float32) / 255.0
        t = torch.from_numpy(x).permute(2, 0, 1)[None].to(self.device)
        return ((t - self.mean) / self.std).to(self.dtype)

    # ---------------------------------------------------------- SigLIP
    def ln(self, x, p, eps):
        return F.layer_norm(x, (x.shape[-1],), self.W[p + '.w'], self.W[p + '.b'], eps)

    def embed(self, px):
        """Patch embedding. A 16 x 16 convolution with stride 16 equals cutting the image into 16 x 16 squares and
        applying one linear layer, which is what we do: same numbers, but no cuDNN (cuDNN alone can take
        several hundred MB on a Jetson Nano)."""
        W, P, g = self.W, self.v['patch'], self.grid
        B, C = px.shape[0], px.shape[1]
        x = px.reshape(B, C, g, P, g, P).permute(0, 2, 4, 1, 3, 5).reshape(B, g * g, C * P * P)   # [1, 1024, 768]
        w = W['v.patch.w'].reshape(W['v.patch.w'].shape[0], -1)                                      # [768, 3*16*16]
        return F.linear(x, w, W['v.patch.b']) + W['v.pos'][None]                                     # [1, 1024, 768]

    def vit_layer(self, i, h, bias=None):
        W, p, eps = self.W, 'v.L%d.' % i, self.v['ln_eps']
        B, N, D = h.shape
        H = self.v['heads']
        hd = D // H
        x = self.ln(h, p + 'ln1', eps)
        q = F.linear(x, W[p + 'q.w'], W[p + 'q.b']).view(B, N, H, hd).transpose(1, 2)
        k = F.linear(x, W[p + 'k.w'], W[p + 'k.b']).view(B, N, H, hd).transpose(1, 2)
        v = F.linear(x, W[p + 'v.w'], W[p + 'v.b']).view(B, N, H, hd).transpose(1, 2)
        s = torch.matmul(q, k.transpose(-1, -2)) * (hd ** -0.5)                    # [1, 12, N, N]
        if bias is not None:
            s = s + bias                                                              # log(size) of each key token
        a = torch.softmax(s.float(), dim=-1).to(q.dtype)
        o = torch.matmul(a, v).transpose(1, 2).reshape(B, N, D)
        h = h + F.linear(o, W[p + 'o.w'], W[p + 'o.b'])
        x = self.ln(h, p + 'ln2', eps)
        return h + F.linear(gelu_tanh(F.linear(x, W[p + 'fc1.w'], W[p + 'fc1.b'])), W[p + 'fc2.w'], W[p + 'fc2.b'])

    def merge(self, h):
        """Every non-anchor patch joins its most similar anchor within the 7 x 7 window; average each group."""
        a, o = self.anchors, self.others
        hn = F.normalize(h[0].float(), dim=-1)
        sim = hn[o] @ hn[a].t()                                                       # [768, 256] cosine
        local = sim.masked_fill(self.outside, float('-inf'))
        local = torch.where(self.no_local[:, None], sim, local)
        assign = torch.empty(self.N, dtype=torch.long, device=self.device)
        assign[a] = torch.arange(self.M, device=self.device)
        assign[o] = local.argmax(1)
        x = h[0].float()
        ones = torch.ones(self.N, device=self.device)
        sizes = torch.zeros(self.M, device=self.device).index_add_(0, assign, ones)
        mean = torch.zeros(self.M, x.shape[-1], device=self.device).index_add_(0, assign, x) / sizes[:, None]
        sq = torch.zeros(self.M, device=self.device).index_add_(0, assign, (x * x).sum(-1)) / sizes
        spread = torch.log1p((sq - (mean * mean).sum(-1)).clamp(min=0).sqrt() / (mean.norm(dim=-1) + 1e-6))
        return mean[None].to(self.dtype), sizes, assign, spread

    def restore(self, m, sizes, spread):
        W = self.W
        parts = [F.layer_norm(m, (m.shape[-1],), W['r.norm.w'], W['r.norm.b'], 1e-5)]
        if self.meta['foremerge'].get('restore_extra', 0) == 2:
            parts += [torch.log(sizes)[None, :, None].to(m.dtype), spread[None, :, None].to(m.dtype)]
        y = F.gelu(F.linear(torch.cat(parts, -1), W['r.fc1.w'], W['r.fc1.b']))
        return m + F.linear(y, W['r.fc2.w'], W['r.fc2.b'])

    def encode(self, px, mode):
        """Vision tokens on the 32 x 32 grid, [1, 1024, 768]. mode: 'full' or 'foremerge'."""
        h = self.embed(px)
        L = self.v['layers']
        if mode == 'full':
            for i in range(L):
                h = self.vit_layer(i, h)
            return self.ln(h, 'v.post', self.v['ln_eps'])
        for i in range(self.k):
            h = self.vit_layer(i, h)
        m, sizes, assign, spread = self.merge(h)
        m = self.restore(m, sizes, spread)
        bias = torch.log(sizes)[None, None, None, :].to(self.dtype)
        for i in range(self.k, L):
            m = self.vit_layer(i, m, bias)
        return self.ln(m, 'v.post', self.v['ln_eps'])[:, assign, :]

    # ---------------------------------------------------------- connector
    def connector(self, feats, which):
        W, p = self.W, ('cf.' if which == 'full' else 'cm.')
        B, n, d = feats.shape
        s = int(round(n ** 0.5))
        x = F.adaptive_avg_pool2d(feats.transpose(1, 2).reshape(B, d, s, s), (16, 16))
        x = x.flatten(2).transpose(1, 2)                                              # [1, 256, 768]
        return F.linear(F.gelu(F.linear(x, W[p + 'fc1.w'], W[p + 'fc1.b'])), W[p + 'fc2.w'], W[p + 'fc2.b'])

    # ---------------------------------------------------------- SmolLM2 (Llama) with KV cache
    def rope(self, pos0, T):
        pos = torch.arange(pos0, pos0 + T, device=self.device).float()
        f = pos[:, None] * self.inv_freq[None, :]
        e = torch.cat([f, f], -1)
        return e.cos().to(self.dtype)[None, None], e.sin().to(self.dtype)[None, None]

    @staticmethod
    def rot(x):
        h = x.shape[-1] // 2
        return torch.cat([-x[..., h:], x[..., :h]], -1)

    def lm_step(self, x, cache, pos0):
        """x: [1, T, 576] input embeddings. Returns logits of the last position and the updated cache."""
        W, l = self.W, self.l
        B, T, D = x.shape
        Hq, Hk, hd = l['heads'], l['kv_heads'], self.hd
        rep = Hq // Hk
        cos, sin = self.rope(pos0, T)
        mask = None
        if T > 1:
            mask = torch.full((T, T), float('-inf'), device=self.device).triu(1).to(self.dtype)
        h = x
        for i in range(l['layers']):
            p = 'lm.L%d.' % i
            a = rms_norm(h, W[p + 'ln_in'], l['rms_eps'])
            q = F.linear(a, W[p + 'q']).view(B, T, Hq, hd).transpose(1, 2)
            k = F.linear(a, W[p + 'k']).view(B, T, Hk, hd).transpose(1, 2)
            v = F.linear(a, W[p + 'v']).view(B, T, Hk, hd).transpose(1, 2)
            q = q * cos + self.rot(q) * sin
            k = k * cos + self.rot(k) * sin
            if cache[i] is not None:
                k = torch.cat([cache[i][0], k], 2)
                v = torch.cat([cache[i][1], v], 2)
            cache[i] = (k, v)
            kk, vv = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
            s = torch.matmul(q, kk.transpose(-1, -2)) * (hd ** -0.5)
            if mask is not None:
                s = s + mask
            att = torch.softmax(s.float(), dim=-1).to(q.dtype)
            o = torch.matmul(att, vv).transpose(1, 2).reshape(B, T, D)
            h = h + F.linear(o, W[p + 'o'])
            a = rms_norm(h, W[p + 'ln_post'], l['rms_eps'])
            h = h + F.linear(F.silu(F.linear(a, W[p + 'gate'])) * F.linear(a, W[p + 'up']), W[p + 'down'])
        last = rms_norm(h[:, -1:], W['lm.norm'], l['rms_eps'])
        head = W['lm.head'] if 'lm.head' in W else W['lm.embed']
        return F.linear(last, head)[:, -1], cache                                   # only the last position is needed

    def decode(self, ids):
        s = b''.join(self.id_bytes[i] for i in ids if i not in self.special).decode('utf-8', errors='replace')
        if self.meta.get('clean_up_spaces'):
            for a, b in ((' .', '.'), (' ?', '?'), (' !', '!'), (' ,', ','), (" ' ", "'"), (" n't", "n't"),
                         (" 'm", "'m"), (" 's", "'s"), (" 've", "'ve"), (" 're", "'re")):
                s = s.replace(a, b)
        return s.strip()

    def sync(self):
        if self.device == 'cuda':
            torch.cuda.synchronize()

    def caption(self, px, mode, max_new=24):
        """One timed caption. Returns caption, ms to the first token, ms for the whole caption, token count."""
        with torch.no_grad():
            self.sync()
            t0 = time.perf_counter()
            Z = self.connector(self.encode(px, mode), mode)
            seq = torch.cat([Z, self.prompt], 1)
            cache = [None] * self.l['layers']
            logits, cache = self.lm_step(seq, cache, 0)
            if not bool(torch.isfinite(logits).all()):
                raise RuntimeError('non-finite values (float16 overflow?): rerun with --dtype float32')
            nxt = int(logits.argmax(-1))
            self.sync()
            ttft = (time.perf_counter() - t0) * 1e3
            toks, pos = [], seq.shape[1]
            for _ in range(max_new):
                if nxt == self.eos:
                    break
                toks.append(nxt)
                e = self.W['lm.embed'][nxt][None, None]
                logits, cache = self.lm_step(e, cache, pos)
                pos += 1
                nxt = int(logits.argmax(-1))
            self.sync()
            total = (time.perf_counter() - t0) * 1e3
        return self.decode(toks), ttft, total, len(toks)


# ------------------------------------------------------------------ benchmark
def pct(xs, q):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(math.ceil(q / 100.0 * len(xs))) - 1)] if xs else float('nan')


def main():
    here = os.path.dirname(os.path.abspath(__file__))
    ap = argparse.ArgumentParser()
    ap.add_argument('--weights', default=os.path.join(here, 'foremerge_jetson_weights.npz'))
    ap.add_argument('--meta', default=os.path.join(here, 'foremerge_jetson_meta.json'))
    ap.add_argument('--images', default=os.path.join(here, 'images'))
    ap.add_argument('--limit', type=int, default=0, help='use only the first N images (0 = all)')
    ap.add_argument('--runs', type=int, default=3, help='timed runs per image and pipeline (median is kept)')
    ap.add_argument('--warmup', type=int, default=3)
    ap.add_argument('--max-new', type=int, default=24)
    ap.add_argument('--dtype', default='float16', choices=['float16', 'float32'])
    ap.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    ap.add_argument('--modes', default='full,foremerge')
    ap.add_argument('--out', default=os.path.join(here, 'jetson_results'))
    a = ap.parse_args()
    modes = [m.strip() for m in a.modes.split(',') if m.strip()]
    dtype = torch.float16 if a.dtype == 'float16' else torch.float32

    files = sorted(f for f in glob.glob(os.path.join(a.images, '*')) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    if a.limit:
        files = files[:a.limit]
    if not files:
        raise SystemExit('no .jpg/.png images found in %s' % a.images)
    _, avail0 = mem_mb()
    print('torch %s | device %s | %s | %d images | %d runs each | available memory before loading: %.0f MB'
          % (torch.__version__, a.device, a.dtype, len(files), a.runs, avail0))
    t = time.time()
    fm = ForeMergeJetson(a.weights, a.meta, a.device, dtype)
    peak, avail = mem_mb()
    print('model loaded in %.1f s | process peak %.0f MB | available now %.0f MB' % (time.time() - t, peak, avail))

    px0 = fm.pixels(Image.open(files[0]))
    for m in modes:
        for _ in range(a.warmup):
            fm.caption(px0, m, a.max_new)
    print('warm-up done')

    rows, ref = [], fm.meta.get('reference', {})
    gpu_peak = dict((m, 0.0) for m in modes)
    for n, f in enumerate(files):
        img = Image.open(f)
        t = time.perf_counter()
        px = fm.pixels(img)
        prep = (time.perf_counter() - t) * 1e3
        res = dict((m, []) for m in modes)
        for _ in range(a.runs):                                   # round-robin: drift hits both pipelines equally
            for m in modes:
                if a.device == 'cuda':
                    torch.cuda.reset_peak_memory_stats()
                res[m].append(fm.caption(px, m, a.max_new))
                if a.device == 'cuda':
                    gpu_peak[m] = max(gpu_peak[m], torch.cuda.max_memory_allocated() / 2 ** 20)
        name = os.path.basename(f)
        for m in modes:
            cap = res[m][0][0]
            row = {'image': name, 'mode': m, 'caption': cap,
                   'ttft_ms': float(np.median([r[1] for r in res[m]])),
                   'total_ms': float(np.median([r[2] for r in res[m]])),
                   'tokens': res[m][0][3], 'prep_ms': prep,
                   'matches_reference': (cap == ref[name][m]) if name in ref and m in ref[name] else ''}
            rows.append(row)
        print('[%d/%d] %s | ' % (n + 1, len(files), name) +
              ' | '.join('%s %.0f ms: %s' % (m, rows[-len(modes) + j]['ttft_ms'], rows[-len(modes) + j]['caption'])
                         for j, m in enumerate(modes)))

    summary = {'device': a.device, 'dtype': a.dtype, 'torch': torch.__version__, 'images': len(files), 'runs': a.runs,
               'max_new': a.max_new, 'available_mb_before_load': avail0, 'process_peak_mb': mem_mb()[0], 'modes': {}}
    for m in modes:
        tt = [r['ttft_ms'] for r in rows if r['mode'] == m]
        to = [r['total_ms'] for r in rows if r['mode'] == m]
        mt = [r['matches_reference'] for r in rows if r['mode'] == m and r['matches_reference'] != '']
        summary['modes'][m] = {'ttft_median_ms': float(np.median(tt)), 'ttft_p95_ms': pct(tt, 95),
                               'total_median_ms': float(np.median(to)), 'total_p95_ms': pct(to, 95),
                               'gpu_peak_mb': gpu_peak[m] if a.device == 'cuda' else float('nan'),
                               'reference_match': ('%d/%d' % (sum(mt), len(mt))) if mt else 'no reference'}
    if 'full' in summary['modes'] and 'foremerge' in summary['modes']:
        f_, g_ = summary['modes']['full'], summary['modes']['foremerge']
        summary['ttft_change_pct'] = 100.0 * (g_['ttft_median_ms'] / f_['ttft_median_ms'] - 1)
        summary['total_change_pct'] = 100.0 * (g_['total_median_ms'] / f_['total_median_ms'] - 1)

    import csv
    with open(a.out + '.csv', 'w') as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        for r in rows:
            w.writerow(r)
    json.dump(summary, open(a.out + '.json', 'w'), indent=1)

    print('\n' + '=' * 92)
    print('JETSON RESULT (%s, %s, batch 1, %d images x %d runs, median)' % (a.device, a.dtype, len(files), a.runs))
    print('=' * 92)
    print('%-11s %12s %10s %14s %10s %12s %16s' % ('pipeline', 'TTFT ms', 'p95', 'caption ms', 'p95', 'GPU peak MB', 'matches ref'))
    for m in modes:
        s = summary['modes'][m]
        print('%-11s %12.1f %10.1f %14.1f %10.1f %12.0f %16s' % (m, s['ttft_median_ms'], s['ttft_p95_ms'], s['total_median_ms'],
                                                                s['total_p95_ms'], s['gpu_peak_mb'], s['reference_match']))
    if 'ttft_change_pct' in summary:
        print('ForeMerge vs full: time to first word %+.1f%%, whole caption %+.1f%%'
              % (summary['ttft_change_pct'], summary['total_change_pct']))
    print('process peak memory %.0f MB | written %s.csv and %s.json' % (summary['process_peak_mb'], a.out, a.out))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/export_for_jetson.py
"""Run ONCE on a PC (your laptop or Kaggle) that has transformers installed.

Packs everything the Jetson needs into two files, so the Jetson never needs transformers:
    foremerge_jetson_weights.npz   SigLIP vision encoder, SmolLM2-135M, both connectors and R, in float16
    foremerge_jetson_meta.json     sizes of every model, the prompt tokens, the tokenizer's byte table,
                                   and (optionally) reference captions to check the Jetson against

Usage (from the folder that holds this script):
    python export_for_jetson.py --weights-dir ..\\foremerge_app\\weights --out jetson_package
    python export_for_jetson.py --weights-dir weights --out jetson_package --ref-images images --ref-count 10

--weights-dir must hold long_M256.pt, conn7_C_s0.pt (or conn_C.pt) and restore6_content.pt
(or restore7_content_s0.pt / restore4_L3_s0.pt): the same three files the Streamlit demo uses.
--ref-images (optional): the photos you will also copy to the Jetson. Reference captions are made with
foremerge.py (it must sit next to this script or in ../foremerge_app) so the Jetson run can be checked.
"""
import argparse, json, os, sys
import numpy as np
import torch

VIS_ID, LM_ID = 'google/siglip-base-patch16-512', 'HuggingFaceTB/SmolLM2-135M'
PROMPT = 'Describe this image in one sentence.'
K_CUT, M_TOKENS, RADIUS = 3, 256, 3


def torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def find(folder, names):
    for n in names:
        p = os.path.join(folder, n)
        if os.path.exists(p):
            return p
    raise FileNotFoundError('none of %s found in %s' % (names, folder))


def vision_core(model):
    """transformers 4.x keeps the encoder under .vision_model; 5.x puts it on the model itself."""
    for cand in (getattr(model, 'vision_model', None), model):
        if cand is not None and all(hasattr(cand, a) for a in ('embeddings', 'encoder', 'post_layernorm')):
            return cand
    raise AttributeError('could not find the SigLIP embeddings/encoder in this transformers version')


EXPORT_DTYPE = np.float16          # float16 halves the size; the Jetson can still run it in float32 if needed


def arr(t):
    return t.detach().float().cpu().numpy().astype(EXPORT_DTYPE)


def collect_vision(vm, out):
    e = vm.embeddings
    out['v.patch.w'], out['v.patch.b'] = arr(e.patch_embedding.weight), arr(e.patch_embedding.bias)
    out['v.pos'] = arr(e.position_embedding.weight)
    for i, L in enumerate(vm.encoder.layers):
        p, a = 'v.L%d.' % i, L.self_attn
        out[p + 'ln1.w'], out[p + 'ln1.b'] = arr(L.layer_norm1.weight), arr(L.layer_norm1.bias)
        for nm, lin in (('q', a.q_proj), ('k', a.k_proj), ('v', a.v_proj), ('o', a.out_proj)):
            out[p + nm + '.w'], out[p + nm + '.b'] = arr(lin.weight), arr(lin.bias)
        out[p + 'ln2.w'], out[p + 'ln2.b'] = arr(L.layer_norm2.weight), arr(L.layer_norm2.bias)
        out[p + 'fc1.w'], out[p + 'fc1.b'] = arr(L.mlp.fc1.weight), arr(L.mlp.fc1.bias)
        out[p + 'fc2.w'], out[p + 'fc2.b'] = arr(L.mlp.fc2.weight), arr(L.mlp.fc2.bias)
    out['v.post.w'], out['v.post.b'] = arr(vm.post_layernorm.weight), arr(vm.post_layernorm.bias)
    return len(vm.encoder.layers)


def collect_lm(lm, out):
    base = lm.model
    out['lm.embed'] = arr(base.embed_tokens.weight)
    for i, L in enumerate(base.layers):
        p, a, m = 'lm.L%d.' % i, L.self_attn, L.mlp
        for nm, lin in (('q', a.q_proj), ('k', a.k_proj), ('v', a.v_proj), ('o', a.o_proj),
                        ('gate', m.gate_proj), ('up', m.up_proj), ('down', m.down_proj)):
            if getattr(lin, 'bias', None) is not None:
                raise ValueError('unexpected bias in %s%s: this script assumes the SmolLM2 (Llama) layout' % (p, nm))
            out[p + nm] = arr(lin.weight)
        out[p + 'ln_in'] = arr(L.input_layernorm.weight)
        out[p + 'ln_post'] = arr(L.post_attention_layernorm.weight)
    out['lm.norm'] = arr(base.norm.weight)
    tied = lm.lm_head.weight.data_ptr() == base.embed_tokens.weight.data_ptr()
    if not tied:
        out['lm.head'] = arr(lm.lm_head.weight)
    return len(base.layers), tied


def collect_connector(sd, prefix, out):
    """PoolConnector: proj.0 = Linear(768, 576), proj.2 = Linear(576, 576)."""
    sd = sd['connector'] if isinstance(sd, dict) and 'connector' in sd else sd
    out[prefix + 'fc1.w'], out[prefix + 'fc1.b'] = arr(sd['proj.0.weight']), arr(sd['proj.0.bias'])
    out[prefix + 'fc2.w'], out[prefix + 'fc2.b'] = arr(sd['proj.2.weight']), arr(sd['proj.2.bias'])


def collect_restore(sd, out):
    for k in ('norm.weight', 'norm.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'):
        out['r.' + k.replace('weight', 'w').replace('bias', 'b')] = arr(sd[k])
    return int(sd['fc1.weight'].shape[1]) - int(sd['norm.weight'].shape[0])


def bytes_to_unicode():
    """GPT-2 byte-level BPE table (byte value -> printable character)."""
    bs = list(range(ord('!'), ord('~') + 1)) + list(range(ord('\xa1'), ord('\xac') + 1)) + list(range(ord('\xae'), ord('\xff') + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))


def rope_theta(lc):
    """transformers 4.x: config.rope_theta; 5.x: config.rope_parameters['rope_theta']. Only plain RoPE is supported."""
    rp = getattr(lc, 'rope_parameters', None) or {}
    rs = getattr(lc, 'rope_scaling', None) or {}
    theta = getattr(lc, 'rope_theta', None) or rp.get('rope_theta')
    if theta is None:
        raise ValueError('cannot find rope_theta in the SmolLM2 config')
    rtype = rs.get('rope_type') or rs.get('type') or rp.get('rope_type') or 'default'
    if rtype != 'default':
        raise ValueError('rope type %s is not supported by the Jetson script' % rtype)
    return float(theta)


def resize_hw(proc, default):
    size = getattr(proc, 'size', None)
    for get in (lambda s: (s['height'], s['width']), lambda s: (s.height, s.width)):
        try:
            h, w = get(size)
            if h and w:
                return [int(h), int(w)]
        except Exception:
            pass
    return [default, default]


def clean_up_spaces(s):
    """The same replacements as transformers' clean_up_tokenization (used only if the tokenizer has it on)."""
    for a, b in ((' .', '.'), (' ?', '?'), (' !', '!'), (' ,', ','), (" ' ", "'"), (" n't", "n't"),
                 (" 'm", "'m"), (" 's", "'s"), (" 've", "'ve"), (" 're", "'re")):
        s = s.replace(a, b)
    return s


def tokenizer_tables(tok, vocab_size):
    """id -> raw bytes (hex), so the Jetson can turn generated ids into text without the tokenizers library."""
    clean = bool(getattr(tok, 'clean_up_tokenization_spaces', False))
    dec = {v: k for k, v in bytes_to_unicode().items()}
    special = set(int(i) for i in tok.all_special_ids)
    hexes, extra_special = [], []
    for i in range(vocab_size):
        t = tok.convert_ids_to_tokens(i)
        if t is None or i in special or any(ch not in dec for ch in t):
            hexes.append('')
            if i not in special:
                extra_special.append(i)
            continue
        hexes.append(bytes(dec[ch] for ch in t).hex())
    def our_decode(ids):
        s = b''.join(bytes.fromhex(hexes[i]) for i in ids).decode('utf-8', errors='replace')
        return (clean_up_spaces(s) if clean else s).strip()
    tests = ['A dog runs on the grass.', 'Two people ride bicycles near a lake, café, 3 kids!', PROMPT,
             "a dog , a cat . it's here !"]
    for s in tests:
        ids = tok(s, add_special_tokens=False).input_ids
        a, b = our_decode(ids), tok.decode(ids, skip_special_tokens=True).strip()
        if a != b:
            raise RuntimeError('byte decoder mismatch: %r vs %r' % (a, b))
    return hexes, sorted(special | set(extra_special)), clean


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--weights-dir', default='weights')
    ap.add_argument('--out', default='jetson_package')
    ap.add_argument('--ref-images', default='')
    ap.add_argument('--ref-count', type=int, default=10)
    ap.add_argument('--ref-device', default='cpu', help='device for the float32 reference captions (cpu or cuda)')
    a = ap.parse_args()
    from transformers import SiglipVisionModel, AutoModelForCausalLM, AutoTokenizer, AutoImageProcessor
    import transformers
    os.makedirs(a.out, exist_ok=True)

    print('loading %s and %s (transformers %s) ...' % (VIS_ID, LM_ID, transformers.__version__))
    vision = SiglipVisionModel.from_pretrained(VIS_ID).float().eval()
    try:
        lm = AutoModelForCausalLM.from_pretrained(LM_ID, dtype=torch.float32).eval()
    except TypeError:
        lm = AutoModelForCausalLM.from_pretrained(LM_ID, torch_dtype=torch.float32).eval()
    tok = AutoTokenizer.from_pretrained(LM_ID)
    proc = AutoImageProcessor.from_pretrained(VIS_ID)

    out = {}
    vm = vision_core(vision)
    n_vis = collect_vision(vm, out)
    n_lm, tied = collect_lm(lm, out)
    collect_connector(torch_load(find(a.weights_dir, ['long_M256.pt'])), 'cf.', out)
    collect_connector(torch_load(find(a.weights_dir, ['conn7_C_s0.pt', 'conn_C.pt'])), 'cm.', out)
    r_path = find(a.weights_dir, ['restore6_content.pt', 'restore7_content_s0.pt', 'restore4_L3_s0.pt'])
    r_extra = collect_restore(torch_load(r_path), out)

    vc, lc = vision.config, lm.config
    hexes, special, clean = tokenizer_tables(tok, lc.vocab_size)
    meta = {
        'format': 1, 'prompt': PROMPT,
        'prompt_ids': [int(i) for i in tok(PROMPT, return_tensors='pt').input_ids[0].tolist()],
        'eos_id': int(tok.eos_token_id), 'special_ids': special, 'id_bytes_hex': hexes, 'clean_up_spaces': clean,
        'vision': {'hidden': vc.hidden_size, 'heads': vc.num_attention_heads, 'layers': n_vis,
                   'ln_eps': vc.layer_norm_eps, 'image_size': vc.image_size, 'patch': vc.patch_size,
                   'act': vc.hidden_act, 'mean': list(getattr(proc, 'image_mean', [0.5, 0.5, 0.5])),
                   'std': list(getattr(proc, 'image_std', [0.5, 0.5, 0.5])),
                   'resize': resize_hw(proc, vc.image_size)},
        'lm': {'hidden': lc.hidden_size, 'layers': n_lm, 'heads': lc.num_attention_heads,
               'kv_heads': lc.num_key_value_heads, 'rms_eps': lc.rms_norm_eps,
               'rope_theta': rope_theta(lc), 'vocab': lc.vocab_size, 'tied': tied},
        'foremerge': {'k': K_CUT, 'M': M_TOKENS, 'radius': RADIUS, 'restore_extra': r_extra,
                      'restore_file': os.path.basename(r_path)},
        'exported_with': {'transformers': transformers.__version__, 'torch': torch.__version__},
    }
    if vc.hidden_act not in ('gelu_pytorch_tanh', 'gelu_new'):
        raise ValueError('unexpected SigLIP activation %s' % vc.hidden_act)

    wpath = os.path.join(a.out, 'foremerge_jetson_weights.npz')
    np.savez(wpath, **out)
    print('wrote %s (%.0f MB, %d tensors)' % (wpath, os.path.getsize(wpath) / 2 ** 20, len(out)))

    if a.ref_images:
        here = os.path.dirname(os.path.abspath(__file__))
        for p in (here, os.path.join(here, '..', 'foremerge_app')):
            if os.path.exists(os.path.join(p, 'foremerge.py')):
                sys.path.insert(0, p)
                break
        from foremerge import ForeMerge
        from PIL import Image
        fm = ForeMerge(a.weights_dir, device=a.ref_device)
        files = sorted(f for f in os.listdir(a.ref_images) if f.lower().endswith(('.jpg', '.jpeg', '.png')))[:a.ref_count]
        meta['reference'] = {}
        for f in files:
            img = Image.open(os.path.join(a.ref_images, f)).convert('RGB')
            meta['reference'][f] = {m: fm.caption(img, m, max_new=24, runs=1)['caption'] for m in ('full', 'foremerge')}
            print('  reference', f, meta['reference'][f])
    mpath = os.path.join(a.out, 'foremerge_jetson_meta.json')
    json.dump(meta, open(mpath, 'w'))
    print('wrote %s' % mpath)
    print('Copy to the Jetson: jetson_foremerge.py, %s, %s and your images folder.' % (os.path.basename(wpath), os.path.basename(mpath)))


if __name__ == '__main__':
    main()

In [ ]:
import os, sys, io, json, glob, re, math, time, random, zipfile, shutil, socket, urllib.request, importlib
from collections import defaultdict
from multiprocessing import Pool
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageFilter, ImageEnhance
sys.path.insert(0, '/kaggle/working')
import transformers
import export_for_jetson as ex, jetson_foremerge as jf
importlib.reload(ex); importlib.reload(jf)

C = dict(
    out='/kaggle/working', ckpt='/kaggle/working/ckpt', cache='/tmp/cache512', ann_dir='/tmp/annotations',
    pkg='/tmp/smoledgevlm_v2_jetson', zip='/kaggle/working/smoledgevlm_v2_jetson.zip',
    check='/kaggle/working/retrain_check.json',
    vizwiz_ann_url='https://vizwiz.cs.colorado.edu/VizWiz_final/caption/annotations.zip',
    vizwiz_img_url='https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip',
    textcaps_repo='lmms-lab/TextCaps', nocaps_repo='lmms-lab/NoCaps',
    max_cap_tokens=20, aug_p=0.3,
    r_per_source=1000, r_rank=192, r_lr=1e-3, r_bs=8,
    conn_epochs=1, conn_lr=1e-3, conn_micro=16, conn_accum=1, conn_warm_frac=0.025,
    lora_epochs=1, lora_lr=2e-4, lora_batch=16, lora_warm=200, rank=16, alpha=32, dropout=0.05,
    save_every=1000, workers=4, n_dev=300, n_test=500, n_board=100, n_check=500, max_new=40,
    budget_min=690, seed=0,
)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
T0 = time.time()
def minutes():
    return (time.time() - T0) / 60
print(f'torch {torch.__version__} | transformers {transformers.__version__} | GPU visible: {torch.cuda.is_available()}')
if DEV != 'cuda':
    raise SystemExit('No GPU in this session. Settings > Accelerator > GPU T4, then restart the session.')
try:
    socket.create_connection(('huggingface.co', 443), timeout=10).close()
except OSError:
    raise SystemExit('No internet in this session. Settings > Internet: On, then run all again.')
torch.manual_seed(C['seed']); random.seed(C['seed']); np.random.seed(C['seed'])
os.makedirs(C['ckpt'], exist_ok=True)
os.makedirs(C['cache'], exist_ok=True)

def find_input(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

def find_ckpt(name):
    """A finished or partial stage, from this session or from an attached earlier output."""
    here = os.path.join(C['ckpt'], name)
    if os.path.exists(here):
        return here
    hit = find_input(name)
    if hit:
        shutil.copy(hit, here)
        return here
    return None

WFILES = {}
for n in ('long_M256.pt', 'conn7_C_s0.pt', 'restore6_content.pt'):
    WFILES[n] = find_input(n)
assert WFILES['long_M256.pt'], 'attach the dataset made from smoledgevlm_seed0_inputs.zip (long_M256.pt is needed)'
print('current-model files:', WFILES)
st = shutil.disk_usage('/tmp')
print(f'disk /tmp: {st.free / 2**30:.0f} GB free')

## 1. Training data: resize every image once to 512 x 512 (the encoder's input) and keep the captions

In [ ]:
SKIP = {'train2014', 'val2014', 'test2014', 'labels'}
def find_img_dir(split):
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == split and any(f.endswith('.jpg') for f in files[:50]):
            return root
        dirs[:] = [d for d in dirs if d not in SKIP - {split}]
    raise FileNotFoundError(f'no {split} image folder under /kaggle/input: attach the COCO 2014 dataset')
VAL_DIR, TRAIN_DIR = find_img_dir('val2014'), find_img_dir('train2014')

NEED = ('captions_val2014.json', 'captions_train2014.json')
os.makedirs(C['ann_dir'], exist_ok=True)
for n in NEED:
    f = find_input(n)
    if f and not os.path.exists(f"{C['ann_dir']}/{n}"):
        shutil.copy(f, f"{C['ann_dir']}/{n}")
if not all(os.path.exists(f"{C['ann_dir']}/{n}") for n in NEED):
    print('downloading COCO 2014 annotations (~241 MB) ...')
    urllib.request.urlretrieve('http://images.cocodataset.org/annotations/annotations_trainval2014.zip', '/tmp/ann.zip')
    with zipfile.ZipFile('/tmp/ann.zip') as z:
        for n in z.namelist():
            if os.path.basename(n) in NEED:
                with z.open(n) as src, open(f"{C['ann_dir']}/{os.path.basename(n)}", 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    os.remove('/tmp/ann.zip')

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(ex.LM_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

FIRST = re.compile(r'(.+?[.!?])(\s|$)')
def first_sentence(text):
    text = ' '.join(str(text).strip().split())
    if text.isupper():                                       # some VizWiz captions are typed in capitals
        text = text.capitalize()
    m = FIRST.match(text)
    if m:
        return m.group(1)
    return text

def usable_captions(caps):
    """First sentence of every caption; keep those of at most max_cap_tokens tokens, else the shortest one."""
    cleaned = []
    for c in caps:
        s = first_sentence(c)
        if len(s) >= 3:
            cleaned.append(s)
    if not cleaned:
        return []
    short = []
    for s in cleaned:
        if len(tok(s).input_ids) <= C['max_cap_tokens']:
            short.append(s)
    if short:
        return short
    shortest = cleaned[0]
    for s in cleaned:
        if len(s) < len(shortest):
            shortest = s
    return [shortest]

def save512(src, dst):
    """Bicubic resize to 512 x 512, exactly what the encoder's preprocessing does, saved once as JPEG."""
    if os.path.exists(dst):
        return True
    try:
        img = Image.open(src).convert('RGB').resize((512, 512), Image.BICUBIC)
        img.save(dst, quality=90)
        return True
    except Exception as e:
        print('  skipped', dst, e)
        return False

def save512_job(pair):
    return save512(pair[0], pair[1])

MANIFEST_PATH = f"{C['cache']}/manifest.json"
MANIFEST = json.load(open(MANIFEST_PATH)) if os.path.exists(MANIFEST_PATH) else {}

In [ ]:
# 1a. VizWiz-Captions train, first while the disk is empty (its image zip is 11.3 GB). An attached copy is used if there is one.
def download(url, dst):
    """Download with progress; resumes a partial file."""
    done = os.path.getsize(dst) if os.path.exists(dst) else 0
    req = urllib.request.Request(url, headers={'Range': f'bytes={done}-'} if done else {})
    t0, last = time.time(), 0
    with urllib.request.urlopen(req, timeout=60) as r, open(dst, 'ab' if done else 'wb') as f:
        total = int(r.headers.get('Content-Length', 0)) + done
        while True:
            block = r.read(1 << 22)
            if not block:
                break
            f.write(block)
            done += len(block)
            if time.time() - last > 60:
                last = time.time()
                print(f'  {done / 2**30:.1f} of {total / 2**30:.1f} GB ({done / 2**20 / (time.time() - t0):.0f} MB/s)')
    return dst

if 'vizwiz' not in MANIFEST:
    t0 = time.time()
    ann = find_input('train.json') if find_input('VizWiz_train_00000000.jpg') else None
    if ann is None:
        download(C['vizwiz_ann_url'], '/tmp/vizwiz_ann.zip')
        with zipfile.ZipFile('/tmp/vizwiz_ann.zip') as z:
            name = [n for n in z.namelist() if n.endswith('train.json')][0]
            with z.open(name) as src, open('/tmp/vizwiz_train.json', 'wb') as dst:
                shutil.copyfileobj(src, dst)
        ann = '/tmp/vizwiz_train.json'
    raw = json.load(open(ann))
    caps_by_id = defaultdict(list)
    removed = 0
    for a in raw['annotations']:
        if a.get('is_precanned') or a.get('is_rejected'):
            removed += 1
            continue
        caps_by_id[a['image_id']].append(a['caption'])
    names = {im['id']: im['file_name'] for im in raw['images']}
    print(f'VizWiz train: {len(names)} images, {removed} pre-written or rejected captions removed')
    os.makedirs(f"{C['cache']}/vizwiz", exist_ok=True)
    attached = find_input('VizWiz_train_00000000.jpg')
    rows = []
    if attached:
        folder = os.path.dirname(attached)
        jobs = []
        for iid, caps in caps_by_id.items():
            caps = usable_captions(caps)
            if caps:
                dst = f"{C['cache']}/vizwiz/{iid}.jpg"
                jobs.append((os.path.join(folder, names[iid]), dst))
                rows.append({'file': dst, 'caps': caps, 'source': 'vizwiz', 'id': f'vizwiz_{iid}'})
        with Pool(C['workers']) as p:
            ok = p.map(save512_job, jobs, chunksize=64)
        rows = [r for r, good in zip(rows, ok) if good]
    else:
        print('downloading the VizWiz train images (several GB) ...')
        download(C['vizwiz_img_url'], '/tmp/vizwiz_train.zip')
        by_name = {}
        for iid, name in names.items():
            by_name[name] = iid
        with zipfile.ZipFile('/tmp/vizwiz_train.zip') as z:
            for n in z.namelist():
                base = os.path.basename(n)
                if base not in by_name:
                    continue
                iid = by_name[base]
                caps = usable_captions(caps_by_id.get(iid, []))
                if not caps:
                    continue
                dst = f"{C['cache']}/vizwiz/{iid}.jpg"
                try:
                    with z.open(n) as fh:
                        Image.open(io.BytesIO(fh.read())).convert('RGB').resize((512, 512), Image.BICUBIC).save(dst, quality=90)
                except Exception as e:
                    print('  skipped', base, e)
                    continue
                rows.append({'file': dst, 'caps': caps, 'source': 'vizwiz', 'id': f'vizwiz_{iid}'})
        os.remove('/tmp/vizwiz_train.zip')
    MANIFEST['vizwiz'] = rows
    json.dump(MANIFEST, open(MANIFEST_PATH, 'w'))
    print(f"VizWiz train: {len(rows)} images with usable captions in {time.time() - t0:.0f} s")
else:
    print(f"VizWiz train: {len(MANIFEST['vizwiz'])} images (cached)")

print(f'disk /tmp after VizWiz: {shutil.disk_usage("/tmp").free / 2**30:.0f} GB free')

In [ ]:
# 1b. COCO train2014: every image with its 5 captions
if 'coco' not in MANIFEST:
    t0 = time.time()
    raw = json.load(open(f"{C['ann_dir']}/captions_train2014.json"))
    caps_by_id = defaultdict(list)
    for a in raw['annotations']:
        caps_by_id[a['image_id']].append((a['id'], a['caption']))
    names = {im['id']: im['file_name'] for im in raw['images']}
    del raw
    os.makedirs(f"{C['cache']}/coco", exist_ok=True)
    jobs, rows = [], []
    for iid in sorted(caps_by_id):
        src = os.path.join(TRAIN_DIR, names[iid])
        if not os.path.exists(src):
            continue
        caps = usable_captions([c for _, c in sorted(caps_by_id[iid])])
        if not caps:
            continue
        dst = f"{C['cache']}/coco/{iid}.jpg"
        jobs.append((src, dst))
        rows.append({'file': dst, 'caps': caps, 'source': 'coco', 'id': f'coco_{iid}'})
    with Pool(C['workers']) as p:
        ok = p.map(save512_job, jobs, chunksize=64)
    MANIFEST['coco'] = [r for r, good in zip(rows, ok) if good]
    json.dump(MANIFEST, open(MANIFEST_PATH, 'w'))
    print(f"COCO train2014: {len(MANIFEST['coco'])} images in {time.time() - t0:.0f} s")
else:
    print(f"COCO train2014: {len(MANIFEST['coco'])} images (cached)")

In [ ]:
# 1c. TextCaps train (Hugging Face parquet shards, one image per row with its 5 captions)
from huggingface_hub import HfApi, hf_hub_download
import pyarrow.parquet as pq

def hf_shards(repo, prefix):
    files = HfApi().list_repo_files(repo, repo_type='dataset')
    out = []
    for f in files:
        if f.startswith(prefix) and f.endswith('.parquet'):
            out.append(f)
    return sorted(out)

def image_bytes(cell):
    if isinstance(cell, dict):
        return cell.get('bytes')
    return cell

if 'textcaps' not in MANIFEST:
    t0 = time.time()
    os.makedirs(f"{C['cache']}/textcaps", exist_ok=True)
    rows, seen = [], set()
    for shard in hf_shards(C['textcaps_repo'], 'data/train-'):
        local = hf_hub_download(C['textcaps_repo'], shard, repo_type='dataset', cache_dir='/tmp/hf')
        table = pq.read_table(local, columns=['image', 'image_id', 'reference_strs', 'caption_str'])
        for rec in table.to_pylist():
            iid = str(rec['image_id'])
            if iid in seen:
                continue
            seen.add(iid)
            refs = rec.get('reference_strs') or rec.get('caption_str') or []
            caps = usable_captions(refs)
            if not caps:
                continue
            dst = f"{C['cache']}/textcaps/{iid}.jpg"
            if not os.path.exists(dst):
                try:
                    Image.open(io.BytesIO(image_bytes(rec['image']))).convert('RGB').resize((512, 512), Image.BICUBIC).save(dst, quality=90)
                except Exception as e:
                    print('  skipped', iid, e)
                    continue
            rows.append({'file': dst, 'caps': caps, 'source': 'textcaps', 'id': f'textcaps_{iid}'})
        del table
        os.remove(local)                                     # keep the disk free: one shard at a time
        print(f'  {shard}: {len(rows)} images so far ({time.time() - t0:.0f} s)')
    MANIFEST['textcaps'] = rows
    MANIFEST['textcaps_ids'] = sorted(seen)                  # Open Images ids, checked against nocaps below
    json.dump(MANIFEST, open(MANIFEST_PATH, 'w'))
    shutil.rmtree('/tmp/hf', ignore_errors=True)
    print(f"TextCaps train: {len(rows)} images in {time.time() - t0:.0f} s")
else:
    print(f"TextCaps train: {len(MANIFEST['textcaps'])} images (cached)")

In [ ]:
TRAIN = MANIFEST['coco'] + MANIFEST['textcaps'] + MANIFEST['vizwiz']
# The training order must be the same when a later session resumes: keep the image list of the first session.
ids_file = find_ckpt('train_ids.json')
if ids_file:
    saved = json.load(open(ids_file))
    by_id = {}
    for r in TRAIN:
        by_id[r['id']] = r
    kept = []
    for i in saved:
        if i in by_id:
            kept.append(by_id[i])
    print(f'resuming with the first session\'s image list: {len(kept)} of {len(saved)} images found')
    TRAIN = kept
else:
    json.dump([r['id'] for r in TRAIN], open(f"{C['ckpt']}/train_ids.json", 'w'))
for src in ('coco', 'textcaps', 'vizwiz'):
    lens = [len(tok(c).input_ids) for r in MANIFEST[src][:2000] for c in r['caps']]
    print(f"{src:9s} {len(MANIFEST[src]):6d} images | caption tokens (sample): mean {np.mean(lens):.1f}, "
          f"95th percentile {np.percentile(lens, 95):.0f} | example: {MANIFEST[src][0]['caps'][0]}")
print(f'training images in total: {len(TRAIN)} | {minutes():.0f} min so far')

## 2. Models and the merge path (batched version of the training code used for the paper)

In [ ]:
from transformers import SiglipVisionModel, AutoModelForCausalLM
vision = SiglipVisionModel.from_pretrained(ex.VIS_ID).float().eval().to(DEV)
lm = AutoModelForCausalLM.from_pretrained(ex.LM_ID, torch_dtype=torch.float32, attn_implementation='sdpa').eval().to(DEV)
for m_ in (vision, lm):
    for p in m_.parameters():
        p.requires_grad_(False)
VM = ex.vision_core(vision)
LAYERS = VM.encoder.layers
D_V, D_T = vision.config.hidden_size, lm.config.hidden_size
GRID = vision.config.image_size // vision.config.patch_size
N, M, K_CUT, RADIUS = GRID * GRID, ex.M_TOKENS, ex.K_CUT, ex.RADIUS
PROMPT_IDS = tok(ex.PROMPT, return_tensors='pt').input_ids[0].to(DEV)

rr, cc = torch.meshgrid(torch.arange(GRID, device=DEV), torch.arange(GRID, device=DEV), indexing='ij')
COORDS = torch.stack([rr.reshape(-1), cc.reshape(-1)], -1)
_r, _c = COORDS[:, 0], COORDS[:, 1]
_U = torch.where((_r % 2 == 0) & (_c % 2 == 0), 2.0,
                 torch.where((_r % 2 == 1) & (_c % 2 == 1), 1.0, 0.0)).to(DEV) - 1e-6 * torch.arange(N, device=DEV)
ANCHORS = torch.topk(_U, M).indices.sort().values
IS_ANC = torch.zeros(N, dtype=torch.bool, device=DEV)
IS_ANC[ANCHORS] = True
OTHERS = (~IS_ANC).nonzero().squeeze(1)
OUTSIDE = (COORDS[OTHERS][:, None, :] - COORDS[ANCHORS][None, :, :]).abs().max(-1).values > RADIUS
assert not OUTSIDE.all(1).any(), 'every patch must have an anchor within the window'

MEAN = torch.tensor([0.5, 0.5, 0.5], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.5, 0.5, 0.5], device=DEV).view(1, 3, 1, 1)
def normalise(x_uint8):
    return (x_uint8.to(DEV).float() / 255.0 - MEAN) / STD

def run_layer(layer, h, bias=None):
    out = layer(h, attention_mask=bias)
    return out[0] if isinstance(out, (tuple, list)) else out

def layers_to(px, k):
    h = VM.embeddings(px)
    for layer in LAYERS[:k]:
        h = run_layer(layer, h)
    return h

def full_from(hk, k):
    h = hk
    for layer in LAYERS[k:]:
        h = run_layer(layer, h)
    return VM.post_layernorm(h)

def merge_batch(h):
    """Lattice merge for a batch: each non-anchor patch joins its most similar anchor within the 7 x 7 window."""
    B, n, d = h.shape
    hn = F.normalize(h.float(), dim=-1)
    sim = torch.einsum('bod,bad->boa', hn[:, OTHERS], hn[:, ANCHORS])
    sim = sim.masked_fill(OUTSIDE[None], float('-inf'))
    assign = torch.empty(B, n, dtype=torch.long, device=DEV)
    assign[:, ANCHORS] = torch.arange(M, device=DEV)[None].expand(B, -1)
    assign[:, OTHERS] = sim.argmax(-1)
    sizes = torch.zeros(B, M, device=DEV, dtype=h.dtype).scatter_add_(1, assign, torch.ones(B, n, device=DEV, dtype=h.dtype))
    total = torch.zeros(B, M, d, device=DEV, dtype=h.dtype).scatter_add_(1, assign[..., None].expand(-1, -1, d), h)
    return total / sizes[..., None], sizes, assign

def after_cut(hk, mod):
    """Merge + R + log n bias in layers 4-12 + unmerge (the SmolEdgeVLM path)."""
    with torch.no_grad():
        m, sizes, assign = merge_batch(hk)
    if mod is not None:
        m = mod(m)
    bias = torch.log(sizes)[:, None, None, :].to(m.dtype)
    for layer in LAYERS[K_CUT:]:
        m = run_layer(layer, m, bias)
    m = VM.post_layernorm(m)
    return torch.gather(m, 1, assign[..., None].expand(-1, -1, m.shape[-1]))

def pool_to16(feats):
    B, n, d = feats.shape
    s = int(round(n ** 0.5))
    return F.adaptive_avg_pool2d(feats.transpose(1, 2).reshape(B, d, s, s), (16, 16)).flatten(2).transpose(1, 2)

class RestoreV(nn.Module):
    """R: m + fc2(GELU(fc1(LayerNorm(m)))), output layer starts at zero."""
    def __init__(self, d, r):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, r)
        self.fc2 = nn.Linear(r, d)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
    def forward(self, m):
        return m + self.fc2(F.gelu(self.fc1(self.norm(m))))

class PoolConnector(nn.Module):
    """Average pool of the patch grid to 16 x 16 (256 tokens), then Linear 768->576, GELU, Linear 576->576."""
    def __init__(self, d_v, d_t):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_v, d_t), nn.GELU(), nn.Linear(d_t, d_t))
    def forward(self, feats16):
        return self.proj(feats16)

def lr_cosine(s, total, peak, warm):
    if s < warm:
        return peak * (s + 1) / warm
    return peak * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))

# Checks before training: the bias reaches the attention scores, and the batched merge equals the one-image merge.
with torch.no_grad():
    x0 = torch.from_numpy(np.asarray(Image.open(TRAIN[0]['file']), dtype=np.uint8)).permute(2, 0, 1)[None]
    x1 = torch.from_numpy(np.asarray(Image.open(TRAIN[-1]['file']), dtype=np.uint8)).permute(2, 0, 1)[None]
    hk = layers_to(normalise(torch.cat([x0, x1])), K_CUT)
    m0, s0, _ = merge_batch(hk[:1])
    base_out = run_layer(LAYERS[3], m0, None)
    d_zero = (base_out - run_layer(LAYERS[3], m0, torch.zeros(1, 1, 1, M, device=DEV))).abs().max().item()
    d_rand = (base_out - run_layer(LAYERS[3], m0, torch.randn(1, 1, 1, M, device=DEV))).abs().max().item()
    both = after_cut(hk, None)
    one = after_cut(hk[1:], None)
    d_batch = (both[1:] - one).abs().max().item()
print(f'bias check: zero {d_zero:.1e} (~0), random {d_rand:.1e} (> 0) | batch vs single image: {d_batch:.1e} (~0)')
assert d_zero < 1e-4 and d_rand > 1e-3 and d_batch < 1e-3, 'merge path check failed: send this output'

## 3. Data loader with image changes

In [ ]:
def change_image(img, rng):
    """One random change, applied to 30% of training images (darker, blur, noise, JPEG, lower resolution)."""
    if rng.random() >= C['aug_p']:
        return img
    kind = rng.integers(5)
    if kind == 0:
        return ImageEnhance.Brightness(img).enhance(float(rng.uniform(0.3, 0.8)))
    if kind == 1:
        return img.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.8, 2.5))))
    if kind == 2:
        x = np.asarray(img, dtype=np.float32) + rng.normal(0, float(rng.uniform(4, 18)), (512, 512, 3))
        return Image.fromarray(np.clip(x, 0, 255).astype(np.uint8))
    if kind == 3:
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=int(rng.integers(15, 50)))
        buf.seek(0)
        return Image.open(buf).convert('RGB')
    side = int(512 * rng.uniform(0.2, 0.5))
    return img.resize((side, side), Image.BICUBIC).resize((512, 512), Image.BICUBIC)

class TrainImages(torch.utils.data.Dataset):
    """Returns the 512 x 512 image (uint8) and one caption chosen at random."""
    def __init__(self, rows, order, augment):
        self.rows, self.order, self.augment = rows, order, augment
    def __len__(self):
        return len(self.order)
    def __getitem__(self, j):
        r = self.rows[self.order[j]]
        rng = np.random.default_rng((C['seed'] * 1000003 + int(self.order[j]) * 7919 + j) % (2**32))
        img = Image.open(r['file']).convert('RGB')
        if self.augment:
            img = change_image(img, rng)
        cap = r['caps'][int(rng.integers(len(r['caps'])))]
        x = torch.from_numpy(np.asarray(img, dtype=np.uint8).copy()).permute(2, 0, 1)
        return x, cap

def collate(batch):
    xs = []
    caps = []
    for x, c in batch:
        xs.append(x)
        caps.append(c)
    return torch.stack(xs), caps

def loader(order, batch, augment=True):
    ds = TrainImages(TRAIN, order, augment)
    if C['workers'] == 0:
        return torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=False, collate_fn=collate, drop_last=True)
    return torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=False, num_workers=C['workers'],
                                       collate_fn=collate, drop_last=True, prefetch_factor=4)

def pack_caps(caps):
    ids = []
    for c in caps:
        ids.append(tok(c, truncation=True, max_length=C['max_cap_tokens']).input_ids + [tok.eos_token_id])
    Lc = 0
    for x in ids:
        Lc = max(Lc, len(x))
    out = torch.full((len(ids), Lc), tok.pad_token_id, dtype=torch.long, device=DEV)
    msk = torch.zeros((len(ids), Lc), dtype=torch.long, device=DEV)
    for b, x in enumerate(ids):
        out[b, :len(x)] = torch.tensor(x, device=DEV)
        msk[b, :len(x)] = 1
    return out, msk

def caption_loss(model_lm, Z, ids, msk):
    """Input: [visual tokens, prompt, caption, end token]; loss on the caption only (same layout as before)."""
    B, Lc = ids.shape
    emb = model_lm.get_input_embeddings()
    seq = torch.cat([Z, emb(PROMPT_IDS)[None].expand(B, -1, -1), emb(ids)], 1)
    attn = torch.cat([torch.ones(B, Z.shape[1] + PROMPT_IDS.numel(), dtype=torch.long, device=DEV), msk], 1)
    lab = torch.full(seq.shape[:2], -100, dtype=torch.long, device=DEV)
    lab[:, -Lc:] = torch.where(msk.bool(), ids, torch.full_like(ids, -100))
    return model_lm(inputs_embeds=seq, attention_mask=attn, labels=lab).loss

def out_of_time(margin_min):
    return minutes() + margin_min > C['budget_min']

## 4. Stage 1: R, label-free (unchanged recipe, more varied images)

In [ ]:
R_FILE = find_ckpt('r_v2.pt')
R = RestoreV(D_V, C['r_rank']).to(DEV)
if R_FILE:
    R.load_state_dict(torch.load(R_FILE, map_location=DEV)['state'])
    print('R loaded from', R_FILE)
else:
    rng = random.Random(C['seed'])
    pool = []
    for src in ('coco', 'textcaps', 'vizwiz'):
        pool += rng.sample(MANIFEST[src], min(C['r_per_source'], len(MANIFEST[src])))
    torch.manual_seed(C['seed'])
    opt = torch.optim.AdamW(R.parameters(), lr=C['r_lr'], weight_decay=0.01)
    bs = C['r_bs']
    steps = max(1, len(pool) // bs)
    order = rng.sample(range(len(pool)), len(pool))
    aug_rng = np.random.default_rng(C['seed'])
    hist, t0 = [], time.time()
    R.train()
    for s in range(steps):
        for g in opt.param_groups:
            g['lr'] = lr_cosine(s, steps, C['r_lr'], max(1, steps // 20))
        opt.zero_grad(set_to_none=True)
        acc = 0.0
        for j in order[s * bs:(s + 1) * bs]:
            img = change_image(Image.open(pool[j]['file']).convert('RGB'), aug_rng)
            x = torch.from_numpy(np.asarray(img, dtype=np.uint8).copy()).permute(2, 0, 1)[None]
            with torch.no_grad():
                hk = layers_to(normalise(x), K_CUT)
                target = full_from(hk, K_CUT)
            U = after_cut(hk, R)
            pu, pt = pool_to16(U), pool_to16(target)
            loss = ((U - target).pow(2).mean() / target.var() + (pu - pt).pow(2).mean() / pt.var()) / bs
            loss.backward()
            acc += loss.item()
        torch.nn.utils.clip_grad_norm_(R.parameters(), 1.0)
        opt.step()
        hist.append(acc)
        if s == 0 or (s + 1) % 75 == 0 or s == steps - 1:
            print(f'  R step {s + 1:>4}/{steps}  distill {np.mean(hist[-20:]):.4f}  ({time.time() - t0:.0f}s)')
    R.eval()
    torch.save({'state': R.state_dict(), 'hist': hist, 'images': [r['id'] for r in pool], 'seconds': time.time() - t0},
               f"{C['ckpt']}/r_v2.pt")
    R_FILE = f"{C['ckpt']}/r_v2.pt"
    print(f'R trained in {time.time() - t0:.0f} s')
for p in R.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def visual_features(x_uint8):
    """Frozen part, run in float16: encoder to layer 3, merge + R, layers 4-12, unmerge, pool to 16 x 16."""
    with torch.autocast('cuda', dtype=torch.float16):
        hk = layers_to(normalise(x_uint8), K_CUT)
        U = after_cut(hk, R)
        return pool_to16(U).float()

## 5. Stage 2: connector on all training images (checkpoint every 1,000 steps)

In [ ]:
def run_stage(name, params, model_lm, make_inputs, epochs, lr, warm, batch, accum, extra_state):
    """One training stage with resume. make_inputs(x, caps) returns (Z, ids, msk)."""
    final = find_ckpt(f'{name}_final.pt')
    if final:
        return torch.load(final, map_location='cpu'), True
    steps_per_epoch = len(TRAIN) // (batch * accum)
    total = epochs * steps_per_epoch
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01 if name == 'conn' else 0.0)
    scaler = torch.amp.GradScaler('cuda')
    step, hist, seconds = 0, [], 0.0
    part = find_ckpt(f'{name}_part.pt')
    if part:
        ck = torch.load(part, map_location='cpu')
        extra_state['load'](ck['model'])
        opt.load_state_dict(ck['opt'])
        scaler.load_state_dict(ck['scaler'])
        step, hist, seconds = ck['step'], ck['hist'], ck['seconds']
        print(f'{name}: resuming at step {step}/{total}')
    t0 = time.time()
    start_step = step
    stopped = False
    while step < total and not stopped:
        epoch = step // steps_per_epoch
        order = np.random.default_rng(C['seed'] + 100 * epoch + (1 if name == 'lora' else 0)).permutation(len(TRAIN))
        start = (step % steps_per_epoch) * batch * accum
        it = iter(loader(order[start:], batch))
        for _ in range(step % steps_per_epoch, steps_per_epoch):
            for g in opt.param_groups:
                g['lr'] = lr_cosine(step, total, lr, warm)
            opt.zero_grad(set_to_none=True)
            acc = 0.0
            for _a in range(accum):
                x, caps = next(it)
                Z, ids, msk = make_inputs(x, caps)
                with torch.autocast('cuda', dtype=torch.float16):
                    loss = caption_loss(model_lm, Z, ids, msk) / accum
                scaler.scale(loss).backward()
                acc += loss.item()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            hist.append(acc)
            step += 1
            if step == 1 or step % 200 == 0:
                rate = (time.time() - t0) / max(1, step - start_step)
                print(f'  {name} step {step}/{total}  loss {np.mean(hist[-100:]):.4f}  '
                      f'({(time.time() - t0) / 60:.0f} min this session, about {(total - step) * rate / 60:.0f} min left)')
            if step % C['save_every'] == 0 or step == total:
                torch.save({'model': extra_state['save'](), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                            'step': step, 'hist': hist, 'seconds': seconds + time.time() - t0}, f"{C['ckpt']}/{name}_part.pt")
                if out_of_time(30):
                    print(f'{name}: stopping at step {step} to stay inside the session limit. Add this output as an '
                          'input and run all again to continue.')
                    stopped = True
                    break
    if stopped:
        return None, False
    out = {'model': extra_state['save'](), 'hist': hist, 'steps': total, 'seconds': seconds + time.time() - t0}
    torch.save(out, f"{C['ckpt']}/{name}_final.pt")
    if os.path.exists(f"{C['ckpt']}/{name}_part.pt"):
        os.remove(f"{C['ckpt']}/{name}_part.pt")
    return out, True

torch.manual_seed(C['seed'])
CONN = PoolConnector(D_V, D_T).to(DEV)
for p in CONN.parameters():
    p.requires_grad_(True)
def conn_inputs(x, caps):
    Z = CONN(visual_features(x))
    ids, msk = pack_caps(caps)
    return Z, ids, msk
steps_conn = C['conn_epochs'] * (len(TRAIN) // (C['conn_micro'] * C['conn_accum']))
CONN.train()
res_conn, done_conn = run_stage('conn', list(CONN.parameters()), lm, conn_inputs, C['conn_epochs'], C['conn_lr'],
                                max(1, int(C['conn_warm_frac'] * steps_conn)), C['conn_micro'], C['conn_accum'],
                                {'save': lambda: CONN.state_dict(), 'load': CONN.load_state_dict})
if done_conn:
    CONN.load_state_dict(res_conn['model'])
CONN.eval()
for p in CONN.parameters():
    p.requires_grad_(False)
print('connector', 'done' if done_conn else 'not finished', f'| {minutes():.0f} min so far')

## 6. Stage 3: LoRA on SmolLM2-135M (connector, R and encoder frozen)

In [ ]:
from peft import LoraConfig, get_peft_model
import peft.tuners.lora.torchao as _pt
_pt.is_torchao_available = lambda: False
done_lora = False
if done_conn:
    PEFT = get_peft_model(lm, LoraConfig(r=C['rank'], lora_alpha=C['alpha'], lora_dropout=C['dropout'], bias='none',
                                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    LORA_PARAMS = [p for p in PEFT.parameters() if p.requires_grad]
    N_LORA = sum(p.numel() for p in LORA_PARAMS)
    print(f'trainable LoRA parameters: {N_LORA / 1e6:.2f} M')
    def lora_state():
        out = {}
        for k, v in PEFT.state_dict().items():
            if 'lora_' in k:
                out[k] = v.detach().cpu()
        return out
    def lora_load(sd):
        PEFT.load_state_dict(sd, strict=False)
    def lora_inputs(x, caps):
        with torch.no_grad():
            Z = CONN(visual_features(x))
        ids, msk = pack_caps(caps)
        return Z, ids, msk
    PEFT.train()
    res_lora, done_lora = run_stage('lora', LORA_PARAMS, PEFT, lora_inputs, C['lora_epochs'], C['lora_lr'], C['lora_warm'],
                                    C['lora_batch'], 1, {'save': lora_state, 'load': lora_load})
    if done_lora:
        lora_load(res_lora['model'])
    PEFT.eval()
print('LoRA', 'done' if done_lora else 'not finished', f'| {minutes():.0f} min so far')

## 7. Board packages (the same format and script as before) and the comparison with the current model

In [ ]:
assert done_conn and done_lora, 'training is not finished: add this output as an input and run all again'
from transformers import AutoImageProcessor
proc = AutoImageProcessor.from_pretrained(ex.VIS_ID)
lm_plain = AutoModelForCausalLM.from_pretrained(ex.LM_ID, torch_dtype=torch.float32).eval()
vc, lc = vision.config, lm_plain.config
hexes, special, clean = ex.tokenizer_tables(tok, lc.vocab_size)

def build_meta(r_extra, restore_file):
    return {
        'format': 1, 'prompt': ex.PROMPT,
        'prompt_ids': [int(i) for i in tok(ex.PROMPT, return_tensors='pt').input_ids[0].tolist()],
        'eos_id': int(tok.eos_token_id), 'special_ids': special, 'id_bytes_hex': hexes, 'clean_up_spaces': clean,
        'vision': {'hidden': vc.hidden_size, 'heads': vc.num_attention_heads, 'layers': len(LAYERS), 'ln_eps': vc.layer_norm_eps,
                   'image_size': vc.image_size, 'patch': vc.patch_size, 'act': vc.hidden_act,
                   'mean': list(getattr(proc, 'image_mean', [0.5, 0.5, 0.5])), 'std': list(getattr(proc, 'image_std', [0.5, 0.5, 0.5])),
                   'resize': ex.resize_hw(proc, vc.image_size)},
        'lm': {'hidden': lc.hidden_size, 'layers': lc.num_hidden_layers, 'heads': lc.num_attention_heads,
               'kv_heads': lc.num_key_value_heads, 'rms_eps': lc.rms_norm_eps, 'rope_theta': ex.rope_theta(lc),
               'vocab': lc.vocab_size, 'tied': True},
        'foremerge': {'k': ex.K_CUT, 'M': ex.M_TOKENS, 'radius': ex.RADIUS, 'restore_extra': r_extra, 'restore_file': restore_file},
        'exported_with': {'transformers': transformers.__version__, 'torch': torch.__version__},
    }

COMMON = {}
ex.collect_vision(VM, COMMON)
ex.collect_connector(ex.torch_load(WFILES['long_M256.pt']), 'cf.', COMMON)    # full-encoder mode: unchanged
NEW_BASE = dict(COMMON)
_, tied = ex.collect_lm(lm_plain, NEW_BASE)
assert tied
ex.collect_connector(CONN.state_dict(), 'cm.', NEW_BASE)
r_extra = ex.collect_restore(R.state_dict(), NEW_BASE)
META_NEW = build_meta(r_extra, 'r_v2.pt')
META_NEW['training'] = {'data': {s: len(MANIFEST[s]) for s in ('coco', 'textcaps', 'vizwiz')}, 'settings': C}
shutil.rmtree(C['pkg'], ignore_errors=True)
os.makedirs(C['pkg'])
np.savez(f"{C['pkg']}/smoledgevlm_v2_weights.npz", **NEW_BASE)

merged = PEFT.merge_and_unload().float().eval()
NEW_LORA = dict(NEW_BASE)
ex.collect_lm(merged, NEW_LORA)
np.savez(f"{C['pkg']}/smoledgevlm_v2_lora_weights.npz", **NEW_LORA)
json.dump(META_NEW, open(f"{C['pkg']}/smoledgevlm_v2_meta.json", 'w'))
del merged, PEFT, lm, NEW_LORA
torch.cuda.empty_cache()

MODELS = {}
MODELS['v2'] = jf.ForeMergeJetson(f"{C['pkg']}/smoledgevlm_v2_weights.npz", f"{C['pkg']}/smoledgevlm_v2_meta.json", DEV, torch.float16)
MODELS['v2 + LoRA'] = jf.ForeMergeJetson(f"{C['pkg']}/smoledgevlm_v2_lora_weights.npz", f"{C['pkg']}/smoledgevlm_v2_meta.json", DEV, torch.float16)
if WFILES['conn7_C_s0.pt'] and WFILES['restore6_content.pt']:
    OLD = dict(COMMON)
    ex.collect_lm(lm_plain, OLD)
    ex.collect_connector(ex.torch_load(WFILES['conn7_C_s0.pt']), 'cm.', OLD)
    old_extra = ex.collect_restore(ex.torch_load(WFILES['restore6_content.pt']), OLD)
    os.makedirs('/tmp/old_pkg', exist_ok=True)
    np.savez('/tmp/old_pkg/w.npz', **OLD)
    json.dump(build_meta(old_extra, 'restore6_content.pt'), open('/tmp/old_pkg/m.json', 'w'))
    MODELS['current'] = jf.ForeMergeJetson('/tmp/old_pkg/w.npz', '/tmp/old_pkg/m.json', DEV, torch.float16)
    del OLD
old_lora = find_input('foremerge_lora_weights.npz')
if old_lora is None and find_input('ours_lora_jetson.zip'):
    with zipfile.ZipFile(find_input('ours_lora_jetson.zip')) as z:
        z.extractall('/tmp/old_lora')
    old_lora = glob.glob('/tmp/old_lora/**/foremerge_lora_weights.npz', recursive=True)[0]
if old_lora and 'current' in MODELS:
    MODELS['current + LoRA'] = jf.ForeMergeJetson(old_lora, '/tmp/old_pkg/m.json', DEV, torch.float16)
del lm_plain, vision
torch.cuda.empty_cache()
print('models for the comparison:', list(MODELS))

In [ ]:
# Comparison sets, 500 images each: COCO held-out (first 100 = board images), nocaps validation, TextCaps validation
raw = json.load(open(f"{C['ann_dir']}/captions_val2014.json"))
vrefs = defaultdict(list)
for a in raw['annotations']:
    vrefs[a['image_id']].append((a['id'], a['caption'].strip()))
vfn = {im['id']: im['file_name'] for im in raw['images']}
del raw
VAL = []
for i in sorted(vrefs):                     # the same images as every earlier run
    f = os.path.join(VAL_DIR, vfn[i]) if i in vfn else ''
    if f and os.path.exists(f) and len(vrefs[i]) >= 5:
        VAL.append({'id': str(i), 'file': f, 'refs': [c for _, c in sorted(vrefs[i])]})
    if len(VAL) >= C['n_dev'] + C['n_test']:
        break
SETS = {'COCO held-out': VAL[C['n_dev']:C['n_dev'] + C['n_test']]}
assert SETS['COCO held-out'][0]['id'] == '4256' and SETS['COCO held-out'][99]['id'] == '5601'

def hf_eval_set(repo, prefix, name, refs_col, id_col, oi_col):
    os.makedirs(f'/tmp/eval/{name}', exist_ok=True)
    rows = []
    for shard in hf_shards(repo, prefix):
        local = hf_hub_download(repo, shard, repo_type='dataset', cache_dir='/tmp/hf')
        for rec in pq.read_table(local, columns=['image', id_col, refs_col] + ([oi_col] if oi_col else [])).to_pylist():
            iid = str(rec[id_col])
            dst = f'/tmp/eval/{name}/{iid}.jpg'
            if not os.path.exists(dst):
                Image.open(io.BytesIO(image_bytes(rec['image']))).convert('RGB').save(dst, quality=90)
            rows.append({'id': iid, 'file': dst, 'refs': list(rec[refs_col]), 'oi': str(rec[oi_col]) if oi_col else iid})
        os.remove(local)
    uniq = {}
    for r in rows:
        uniq.setdefault(r['id'], r)
    rows = sorted(uniq.values(), key=lambda r: r['id'])
    return random.Random(C['seed']).sample(rows, min(C['n_check'], len(rows)))

SETS['nocaps validation'] = hf_eval_set(C['nocaps_repo'], 'data/validation-', 'nocaps', 'annotations_captions', 'image_id', 'image_open_images_id')
SETS['TextCaps validation'] = hf_eval_set(C['textcaps_repo'], 'data/val-', 'textcaps', 'reference_strs', 'image_id', None)
overlap = set(r['oi'] for r in SETS['nocaps validation']) & set(MANIFEST.get('textcaps_ids', []))
print({k: len(v) for k, v in SETS.items()}, '| nocaps images also in TextCaps train:', len(overlap))

In [ ]:
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.cider.cider import Cider
OPEN = re.compile(r'^\s*(in (the|this) image,?|the image (shows|depicts|showcases|features)|this image (shows|depicts))'
                  r'\s*(,\s*)?(we (can )?see|there (is|are)|i can see|we can see)?\s*', re.I)
def fair(s):
    m = re.match(r'(.+?[.!?])(\s|$)', s)
    s = m.group(1) if m else s
    t = OPEN.sub('', s).strip()
    return (t[:1].upper() + t[1:]) if t else s

CAPS, TOKS = defaultdict(dict), defaultdict(dict)
t0 = time.time()
for set_name, rows in SETS.items():
    for r in rows:
        px = None
        for name, model in MODELS.items():
            if px is None:
                px = model.pixels(Image.open(r['file']))
            cap, ttft, total, ntok = model.caption(px, 'foremerge', C['max_new'])
            CAPS[(set_name, name)][r['id']] = cap
            TOKS[(set_name, name)][r['id']] = ntok
    print(f'  {set_name}: captioned ({time.time() - t0:.0f} s)')

ptb = PTBTokenizer()
SCORES, PER, BOOT = {}, {}, {}
for set_name, rows in SETS.items():
    ids = [r['id'] for r in rows]
    gts = ptb.tokenize({r['id']: [{'caption': c} for c in r['refs']] for r in rows})
    for name in MODELS:
        caps = CAPS[(set_name, name)]
        res = ptb.tokenize({i: [{'caption': fair(caps[i]) or 'empty'}] for i in ids})
        c, cp = Cider().compute_score(gts, res)
        m, mp = Meteor().compute_score(gts, res)
        toks = float(np.mean([TOKS[(set_name, name)][i] for i in ids]))
        SCORES[f'{set_name} | {name}'] = {'CIDEr': c, 'METEOR': m, 'tokens': toks,
                                          'board_caption_s_estimate': 1.46 + 0.177 * max(0.0, toks - 1)}
        PER[(set_name, name)] = np.array(cp)
    bs = np.random.default_rng(0).integers(0, len(ids), (10000, len(ids)))
    for new, old in (('v2', 'current'), ('v2 + LoRA', 'current + LoRA'), ('v2 + LoRA', 'v2')):
        if (set_name, new) in PER and (set_name, old) in PER:
            d = PER[(set_name, new)] - PER[(set_name, old)]
            lo, hi = np.percentile(d[bs].mean(1), [2.5, 97.5])
            BOOT[f'{set_name} | {new} minus {old}'] = {'mean': float(d.mean()), 'ci95': [float(lo), float(hi)]}

print('\nfair CIDEr (0 to 1 scale), METEOR, tokens per caption, estimated board caption time (1.46 s + 0.177 s per token)')
for set_name in SETS:
    print(f'\n{set_name}')
    for name in MODELS:
        s = SCORES[f'{set_name} | {name}']
        print(f"  {name:16s} CIDEr {s['CIDEr']:.3f}  METEOR {s['METEOR']:.3f}  tokens {s['tokens']:.1f}  "
              f"about {s['board_caption_s_estimate']:.1f} s on the board")
    for k, v in BOOT.items():
        if k.startswith(set_name):
            print(f"  {k.split(' | ')[1]}: {v['mean']:+.3f} [{v['ci95'][0]:+.3f}, {v['ci95'][1]:+.3f}]")
for set_name, rows in SETS.items():
    for r in rows[:3]:
        parts = []
        for name in MODELS:
            parts.append(f"{name}: {CAPS[(set_name, name)][r['id']]}")
        print(f"{set_name} {r['id']} | " + ' | '.join(parts))

In [ ]:
# Package for the board (both new models share one meta file), trained parts, summary
JREF = MODELS['v2 + LoRA']
ref = {}
for r in SETS['COCO held-out'][:20]:
    px = JREF.pixels(Image.open(r['file']))
    ref[os.path.basename(r['file'])] = {m_: JREF.caption(px, m_, 24)[0] for m_ in ('full', 'foremerge')}
META_NEW['reference'] = ref
META_NEW['reference_note'] = 'reference captions are for smoledgevlm_v2_lora_weights.npz'
json.dump(META_NEW, open(f"{C['pkg']}/smoledgevlm_v2_meta.json", 'w'))
open(f"{C['pkg']}/README.txt", 'w').write(
    'SmolEdgeVLM v2 (COCO + TextCaps + VizWiz) for the Jetson Nano 2 GB. Unzip into ~/Downloads (creates\n'
    '~/Downloads/smoledgevlm_v2_jetson). Uses jetson_foremerge.py, jetson_benchmark.py and live_caption.py from\n'
    '~/Downloads/foremerge. Two weight files, one meta file: smoledgevlm_v2_lora_weights.npz (with LoRA) and\n'
    'smoledgevlm_v2_weights.npz (without).\n')
with zipfile.ZipFile(C['zip'], 'w', zipfile.ZIP_STORED) as z:
    for f in sorted(os.listdir(C['pkg'])):
        z.write(f"{C['pkg']}/{f}", f'smoledgevlm_v2_jetson/{f}')
for f in glob.glob(f"{C['ckpt']}/*_part.pt"):
    os.remove(f)
train_time = {}
for name in ('r_v2', 'conn_final', 'lora_final'):
    train_time[name] = torch.load(f"{C['ckpt']}/{name}.pt", map_location='cpu').get('seconds')
summary = {'data': {s: len(MANIFEST[s]) for s in ('coco', 'textcaps', 'vizwiz')}, 'settings': C,
           'training_seconds': train_time, 'scores': SCORES, 'bootstrap': BOOT,
           'captions': {f'{k[0]} | {k[1]}': v for k, v in CAPS.items()}, 'tokens': {f'{k[0]} | {k[1]}': v for k, v in TOKS.items()}}
json.dump(summary, open(C['check'], 'w'), indent=1)
print('training time (minutes):', {k: round(v / 60, 1) if v else None for k, v in train_time.items()})
print(f"written {C['check']} and {C['zip']} ({os.path.getsize(C['zip']) / 2**20:.0f} MB) | total {minutes():.0f} min")